### [ML in Finance: Time Series Momentum Analysis](https://medium.com/financial-engineering/ml-in-finance-series-time-series-momentum-analysis-in-python-bce455a03ccf)

In [1]:
!pip install -qq yfinance

In [2]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, GridSearchCV, RepeatedKFold
from sklearn.preprocessing import MinMaxScaler

from IPython.display import display

import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)
np.set_printoptions(precision=5, suppress=True)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.6f}".format)

In [3]:
# Getting historical market data from SPY (ETF)
df = yf.download("SPY", start="2000-01-01", end="2025-01-01", progress=False, auto_adjust=True, threads=True)
display(df.sample(15))

Price,Close,High,Low,Open,Volume
Ticker,SPY,SPY,SPY,SPY,SPY
Date,,,,,
2017-08-28,213.691879,214.242330,213.272472,214.216118,40565600
2014-07-21,162.213898,162.345421,161.465873,162.008397,67592000
2024-02-16,487.459503,490.738430,486.717828,489.596673,75532900
2020-01-21,303.591156,304.397560,303.151320,303.224616,77742400
2006-02-16,89.161453,89.195971,88.484934,88.595388,61017900
2013-04-10,127.240089,127.400470,126.005143,126.037215,135711100
2001-03-07,80.831161,80.869354,80.054550,80.780235,6371700
2016-10-06,185.714401,185.938170,184.819315,185.361525,62927400


In [4]:
df["Ret"] = df["Close"].pct_change()

name = "Ret"

df["Ret10_i"] = (df[name].rolling(10).apply(lambda x: 100 * ((np.prod(1 + x)) ** (1 / 10) - 1)))
df["Ret25_i"] = (df[name].rolling(25).apply(lambda x: 100 * ((np.prod(1 + x)) ** (1 / 25) - 1)))
df["Ret60_i"] = (df[name].rolling(60).apply(lambda x: 100 * ((np.prod(1 + x)) ** (1 / 60) - 1)))
df["Ret120_i"] = (df[name].rolling(120).apply(lambda x: 100 * ((np.prod(1 + x)) ** (1 / 120) - 1)))
df["Ret240_i"] = (df[name].rolling(240).apply(lambda x: 100 * ((np.prod(1 + x)) ** (1 / 240) - 1)))
df = df.dropna()

df["Ret25"] = df["Ret25_i"].shift(-25)
df = df.dropna()

display(df.sample(15))

Price,Close,High,Low,Open,Volume,Ret,Ret10_i,Ret25_i,Ret60_i,Ret120_i,Ret240_i,Ret25
Ticker,SPY,SPY,SPY,SPY,SPY,,,,,,,
Date,,,,,,,,,,,,
2018-07-16,248.635880,249.045310,248.190838,248.902921,48201000,-0.000894,0.293210,0.034427,0.071038,-0.004637,0.058464,0.089670
2004-10-27,75.987389,76.135487,74.802614,74.977634,73896000,0.012013,0.119489,0.047420,0.046830,0.014436,0.031584,0.234379
2020-07-02,289.065094,292.277649,288.398511,290.925949,69344200,0.005507,0.058348,0.138085,0.280243,-0.026674,0.027816,0.276805
2024-09-20,559.770386,560.814566,556.736330,559.366530,77503100,-0.001729,0.535311,0.120631,0.073198,0.075754,0.122274,0.075269
2012-01-31,102.567055,103.238743,102.067173,103.113785,157212000,-0.000380,0.152043,0.188842,0.106321,0.102429,0.000700,0.131028
2012-09-20,115.685806,115.748878,114.834188,115.149597,154009800,0.000068,0.202634,0.160339,0.176501,0.038519,0.105570,-0.125237
2022-11-11,381.322906,382.126675,376.634212,378.528829,93839900,0.009679,0.241312,0.376341,-0.111661,0.010340,-0.049478,-0.171563
2008-11-11,65.297653,67.021566,64.482982,66.017772,418498200,-0.030876,-0.433932,-0.431943,-0.585032,-0.357796,-0.197061,0.054007


In [5]:
X, y = df.iloc[:, 0:-1], df.iloc[:, -1]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=int(len(y) * 0.5), shuffle=False
)

lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

# Make predictions
y_pred = lin_reg.predict(X_train)
y_pred_test = lin_reg.predict(X_test)

score = lin_reg.score(X_train, y_train)
print(f"Score: {score:.6f}")

Score: 0.047768


In [6]:
print("In-Sample Performance:")
train_mse = mean_squared_error(y_train, y_pred)
train_r2 = r2_score(y_train, y_pred)
print(f"Mean squared error: {train_mse:.5f}")
print(f"Coefficient of determination (R2): {train_r2:.5f}")

print("\nOut-of-Sample Performance:")
test_mse = mean_squared_error(y_test, y_pred_test)
test_r2 = r2_score(y_test, y_pred_test)
print(f"Mean squared error: {test_mse:.5f}")
print(f"Coefficient of determination (R2): {test_r2:.5f}")

In-Sample Performance:
Mean squared error: 0.05033
Coefficient of determination (R2): 0.04777

Out-of-Sample Performance:
Mean squared error: 0.76791
Coefficient of determination (R2): -21.03935


In [7]:
# Train the model
e_net = ElasticNet(alpha=0.0001, l1_ratio=0.1)
e_net.fit(X_train, y_train)

# Make predictions
y_pred_elastic = e_net.predict(X_test)
elastic_mse = np.mean((y_pred_elastic - y_test) ** 2)
print(f"Mean Squared Error on test set (Elastic Net): {elastic_mse:.5f}")

Mean Squared Error on test set (Elastic Net): 0.60827


In [8]:
model = ElasticNet()
cv = RepeatedKFold(n_splits=10, n_repeats=3, random_state=1)
grid = dict()
grid["alpha"] = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 0.0, 1.0, 10.0, 100.0]
grid["l1_ratio"] = [0, 0.01, 0.1, 0.2, 0.5, 0.7, 1]
search = GridSearchCV(model, grid, scoring="neg_mean_squared_error", cv=cv, n_jobs=-1)
results = search.fit(X_train, y_train)

print(f"Best MSE: {results.best_score_:.3f}")
print(f"Best Config: {results.best_params_}")

Best MSE: -0.051
Best Config: {'alpha': 0.001, 'l1_ratio': 0.2}
